In [2]:
from gr00t.model.gr00t_n1 import GR00T_N1_5

model = GR00T_N1_5.from_pretrained(
        pretrained_model_name_or_path="/home/minh/Workspace/projects/IsaacLabEvalTasks/checkpoints/grootn15",
        tune_visual=False,  # backbone's vision tower
        tune_llm=False,  # backbone's LLM
        tune_projector=True,  # action head's projector
        tune_diffusion_model=True,  # action head's DiT
    )



/home/minh/Workspace/projects/IsaacLabEvalTasks/.venv/lib/python3.11/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Loading pretrained dual brain from /home/minh/Workspace/projects/IsaacLabEvalTasks/checkpoints/grootn15
Tune backbone vision tower: False
Tune backbone LLM: False
Tune action head projector: True
Tune action head DiT: True
Model not found or avail in the huggingface hub. Loading from local path: /home/minh/Workspace/projects/IsaacLabEvalTasks/checkpoints/grootn15
Tune backbone llm: False
Tune backbone visual: True
Total number of DiT parameters:  550386688
Total number of SelfAttentionTransformer parameters:  201433088
Tune action head projector: True
Tune action head diffusion model: True


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Tune backbone llm: False
Tune backbone visual: False
Tune action head projector: True
Tune action head diffusion model: True


In [3]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")


Total parameters:     2,724,163,520
Trainable parameters: 1,068,806,144
Trainable %: 39.2343%


In [4]:
for name, module in model.named_modules():
    trainable = any(p.requires_grad for p in module.parameters(recurse=False))
    if trainable:
        print(name)

# model

action_head.model.timestep_encoder.timestep_embedder.linear_1
action_head.model.timestep_encoder.timestep_embedder.linear_2
action_head.model.transformer_blocks.0.norm1.linear
action_head.model.transformer_blocks.0.attn1.to_q
action_head.model.transformer_blocks.0.attn1.to_k
action_head.model.transformer_blocks.0.attn1.to_v
action_head.model.transformer_blocks.0.attn1.to_out.0
action_head.model.transformer_blocks.0.ff.net.0.proj
action_head.model.transformer_blocks.0.ff.net.2
action_head.model.transformer_blocks.1.norm1.linear
action_head.model.transformer_blocks.1.attn1.to_q
action_head.model.transformer_blocks.1.attn1.to_k
action_head.model.transformer_blocks.1.attn1.to_v
action_head.model.transformer_blocks.1.attn1.to_out.0
action_head.model.transformer_blocks.1.ff.net.0.proj
action_head.model.transformer_blocks.1.ff.net.2
action_head.model.transformer_blocks.2.norm1.linear
action_head.model.transformer_blocks.2.attn1.to_q
action_head.model.transformer_blocks.2.attn1.to_k
action_hea

In [5]:
from gr00t.utils.peft import get_lora_model

model = get_lora_model(
    model,
    rank=8,
    lora_alpha=16,
    lora_dropout=0.01,
    action_head_only=False,
)

trainable params: 4,114,432 || all params: 2,728,277,952 || trainable%: 0.1508


In [6]:
for name, module in model.named_modules():
    trainable = any(p.requires_grad for p in module.parameters(recurse=False))
    if trainable:
        print(name)


base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.k_proj.lora_A.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.k_proj.lora_B.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.v_proj.lora_A.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.v_proj.lora_B.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.q_proj.lora_A.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.0.self_attn.q_proj.lora_B.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.1.self_attn.k_proj.lora_A.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.1.self_attn.k_proj.lora_B.default
base_model.model.backbone.eagle_model.vision_model.vision_model.encoder.layers.1.self_attn.v_pro

In [7]:
target_modules = [
    "attn1.to_q",
    "attn1.to_k",
    "attn1.to_v",
    "attn1.to_out.0",
    "ff.net.0.proj",
    "ff.net.2",
]
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                    # rank
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=target_modules,
    task_type="CAUSAL_LM",  # or CAUSAL_LM if applicable
)
model = get_peft_model(model, lora_config)


/home/minh/Workspace/projects/IsaacLabEvalTasks/.venv/lib/python3.11/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/minh/Workspace/projects/IsaacLabEvalTasks/.venv/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [8]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")


Total parameters:     2,731,423,680
Trainable parameters: 7,260,160
Trainable %: 0.2658%


In [9]:
action_head.model.transformer_blocks

NameError: name 'action_head' is not defined

- action_head.model.transformer_blocks
    - q, k, v, to_out, ff.net.0.proj, ff.net.2 (check coi no1 la cai gi)
- action_head.model.proj_out_1,
- action_head.model.proj_out_2,
- action_head.state_encoder.layer1,
- action_head.state_encoder.layer2,
- action_head.action_encoder.W1,
- action_head.action_encoder.W2,
- action_head.action_encoder.W3,
- action_head.action_decoder.layer1,
- action_head.action_decoder.layer2,
- action_head.vl_transformers:
    - q, k, v, to_out, ff.net.0.proj, ff.net.2 (check coi no1 la cai gi)


In [ ]:
trainable_prefixes = [
    "action_head.model.proj_out_1",
    "action_head.model.proj_out_2",
    "action_head.state_encoder.layer1",
    "action_head.state_encoder.layer2",
    "action_head.action_encoder.W1",
    "action_head.action_encoder.W2",
    "action_head.action_encoder.W3",
    "action_head.action_decoder.layer1",
    "action_head.action_decoder.layer2",
    "backbone.linear",
    "action_head.model.timestep_encoder.timestep_embedder.linear_1",
    "action_head.model.timestep_encoder.timestep_embedder.linear_2",
]
for name, param in model.named_parameters():
    for prefix in trainable_prefixes:
        if prefix in name:
            param.requires_grad = True



In [ ]:
for name, param in model.named_parameters():
    if param.requires_grad == True:
        print(name)

base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_1.weight
base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_1.bias
base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_2.weight
base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_2.bias
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_q.lora_A.default.weight
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_q.lora_B.default.weight
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_k.lora_A.default.weight
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_k.lora_B.default.weight
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_v.lora_A.default.weight
base_model.model.base_model.model.action_head.model.transforme

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")


Total parameters:     2,728,947,648
Trainable parameters: 329,195,520
Trainable %: 12.0631%


In [ ]:
for name, module in model.named_modules():
    trainable = any(p.requires_grad for p in module.parameters(recurse=False))
    if trainable:
        print(name)

# model

base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_1
base_model.model.base_model.model.action_head.model.timestep_encoder.timestep_embedder.linear_2
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_q.lora_A.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_q.lora_B.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_k.lora_A.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_k.lora_B.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_v.lora_A.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_v.lora_B.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_out.0.lora_A.default
base_model.model.base_model.model.action_head.model.transformer_blocks.0.attn1.to_out.0.lora_B.default
base_mod

In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): GR00T_N1_5(
          (backbone): EagleBackbone(
            (eagle_model): Eagle2_5_VLForConditionalGeneration(
              (vision_model): SiglipVisionModel(
                (vision_model): SiglipVisionTransformer(
                  (embeddings): SiglipVisionEmbeddings(
                    (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
                    (position_embedding): Embedding(256, 1152)
                  )
                  (encoder): SiglipEncoder(
                    (layers): ModuleList(
                      (0-26): 27 x SiglipEncoderLayer(
                        (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                        (self_attn): SiglipAttention(
                          (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                     